# Adaa: virtual try-on server (runs on your free Colab GPU)

This notebook lets **Adaa** show *you* wearing the outfits it suggests.

It runs the open-source **CatVTON** model on Google's free GPU, then gives you a **link** and a **secret token** to paste into Adaa → **Preferences → Try-on server**.

**Before you start:** *Runtime → Change runtime type → **T4 GPU*** → Save.

**Then:** *Runtime → Run all*. The first run takes about **5–10 minutes** (downloading the model). At the end you'll see the link and token.

**Keep this tab open** while you use Adaa. When Colab disconnects (after idle time, or the daily free limit), just run it again and paste the new link.

**Privacy:** your photos travel only between your laptop and *your* Colab session. Nothing is stored here after the session ends.

**Licence:** CatVTON is CC BY-NC-SA 4.0, non-commercial use only. That's fine for this student project.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again."
print("GPU ready:", torch.cuda.get_device_name(0))

## 2. Download CatVTON and install what it needs (about 2–3 minutes)

In [ ]:
%cd /content
!test -d CatVTON || git clone --depth 1 https://github.com/Zheng-Chong/CatVTON.git
%cd /content/CatVTON
# Colab already has torch, diffusers, transformers and accelerate.
# Lock the libraries Colab has already loaded (Pillow, numpy...), so installing extras can't swap
# them mid-session (that causes errors like "cannot import name '_Ink' from 'PIL._typing'").
!pip freeze | grep -iE '^(pillow|numpy|scipy|scikit-image|opencv-python|opencv-python-headless|torch|torchvision|numba)==' > /content/keep.txt
!cat /content/keep.txt
# Extras for CatVTON's bundled DensePose/detectron2 code (av, yacs, ...) and our small server:
!pip install -q -c /content/keep.txt av yacs termcolor tabulate cloudpickle hydra-core fvcore iopath omegaconf pycocotools fastapi uvicorn python-multipart nest_asyncio
# rembg: free background remover, used to cut garments out of their photos (hangers, walls)
!pip install -q -c /content/keep.txt "rembg[cpu]"

## 3. Load the try-on model onto the GPU (first time: about 5 minutes)

If this cell shows an **ImportError**, click *Runtime → Restart session*, then *Run all* again (the installs above are kept).

In [ ]:
import os, sys
sys.path.insert(0, "/content/CatVTON")
import torch
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import resize_and_crop, resize_and_padding

WIDTH, HEIGHT = 768, 1024
# The original base model (runwayml/stable-diffusion-inpainting) was removed from Hugging Face;
# this is the mirror CatVTON's own demo uses.
BASE_MODEL = "booksforcharlie/stable-diffusion-inpainting"

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
pipeline = CatVTONPipeline(
    base_ckpt=BASE_MODEL,
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,   # the T4 GPU doesn't support bf16
    use_tf32=True,
    device="cuda",
    # We run the safety filter ourselves (below), so a blocked result is reported clearly
    # instead of being silently replaced with an "NSFW" warning picture.
    skip_safety_check=True,
)

# Safety filter (Stable Diffusion's standard checker). It stays ON. It can block ordinary
# outfits by mistake (short skirts, bare shoulders); Adaa then shows its illustration instead.
import numpy as np
from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker
from transformers import CLIPImageProcessor
safety_checker = StableDiffusionSafetyChecker.from_pretrained(BASE_MODEL, subfolder="safety_checker").to("cuda", dtype=torch.float16)
safety_features = CLIPImageProcessor.from_pretrained(BASE_MODEL, subfolder="feature_extractor")

@torch.no_grad()
def is_blocked(image):
    clip_input = safety_features([image], return_tensors="pt").pixel_values.to("cuda", dtype=torch.float16)
    pixels = np.array(image, dtype=np.float32)[None] / 255.0
    _, flagged = safety_checker(images=pixels, clip_input=clip_input)
    return bool(flagged[0])

mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device="cuda",
)

# ---- Cleaner garment photos = more accurate try-ons ----
from PIL import Image, ImageFilter
# Human-parser label numbers (from CatVTON's SCHP models)
LIP_PARTS = {"upper": [5, 7], "lower": [9, 12], "overall": [5, 6, 7, 9, 10, 12], "shoes": [18, 19]}
ATR_PARTS = {"upper": [4], "lower": [5, 6], "overall": [4, 5, 6, 7], "shoes": [9, 10]}
LIP_BODY = [13, 14, 15, 16, 17]   # face, arms, legs: tells us a person is in the photo

def _labels(parse_img, size):
    if not isinstance(parse_img, Image.Image):
        parse_img = Image.fromarray(np.asarray(parse_img).astype(np.uint8))
    if parse_img.size != size:
        parse_img = parse_img.resize(size, Image.NEAREST)
    return np.array(parse_img)

def find_parts(img, part, min_share=0.01):
    '''(mask of `part`, is a person in the photo?). The mask is None if too little was found.'''
    try:
        lip = _labels(automasker.schp_processor_lip(img), img.size)
        atr = _labels(automasker.schp_processor_atr(img), img.size)
    except Exception as e:
        print("parser skipped:", e)
        return None, False
    person = np.isin(lip, LIP_BODY).mean() > 0.01
    mask = np.isin(lip, LIP_PARTS[part]) | np.isin(atr, ATR_PARTS[part])
    return (mask if mask.mean() > min_share else None), person

try:
    from rembg import remove as remove_background, new_session
    bg_session = new_session("isnet-general-use")
except Exception as e:
    bg_session = None
    print("Background remover unavailable (garments will be used as photographed):", e)

def clean_garment(img, part):
    '''Cut the garment out of its photo and centre it on white.
    Worn by someone -> keep only that garment; on a hanger/table -> remove the background;
    if neither works -> use the photo as it is.'''
    try:
        mask, person = find_parts(img, part, min_share=0.002)
        if person and mask is not None:
            alpha, how = Image.fromarray((mask * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(5)), "worn"
        elif bg_session is not None:
            alpha, how = remove_background(img, session=bg_session).getchannel("A"), "cutout"
        else:
            return img, "original"
        box = alpha.point(lambda a: 255 if a > 40 else 0).getbbox()
        if not box:
            return img, "original"
        white = Image.new("RGB", img.size, "white")
        white.paste(img, mask=alpha)
        cut = white.crop(box)
        pad = int(0.08 * max(cut.size))
        canvas = Image.new("RGB", (cut.width + 2 * pad, cut.height + 2 * pad), "white")
        canvas.paste(cut, (pad, pad))
        return canvas, how
    except Exception as e:
        print("garment clean-up skipped:", e)
        return img, "original"

print("Model loaded.")

## 4. Quick self-test (optional, about 30 seconds)

In [ ]:
from PIL import Image
import time
person = Image.new("RGB", (WIDTH, HEIGHT), (210, 200, 190))
cloth = Image.new("RGB", (WIDTH, HEIGHT), (40, 120, 70))
t = time.time()
try:
    mask = automasker(person, "upper")["mask"]
    print("Masking works.", round(time.time() - t, 1), "s")
except Exception as e:
    print("Masking failed on the blank test image (that can be normal, as there's no person in it):", e)

## 5. Start the try-on service and get your link + token

In [ ]:
import io, secrets, threading, time, re, subprocess
import nest_asyncio, uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from PIL import Image

TOKEN = secrets.token_urlsafe(16)
gpu_lock = threading.Lock()   # one try-on at a time on the GPU
app = FastAPI()


def check(token):
    if token != TOKEN:
        raise HTTPException(status_code=401, detail="Wrong token")


@app.get("/health")
def health(x_adaa_token: str = Header(default="")):
    check(x_adaa_token)
    return {"ok": True, "model": "CatVTON", "gpu": torch.cuda.get_device_name(0)}


@app.post("/tryon")
def tryon(
    person: UploadFile = File(...),
    cloth: UploadFile = File(...),
    cloth_type: str = Form("upper"),        # upper | lower | overall | shoes (experimental)
    steps: int = Form(30),
    seed: int = Form(42),
    x_adaa_token: str = Header(default=""),
):
    check(x_adaa_token)
    if cloth_type not in ("upper", "lower", "overall", "shoes"):
        raise HTTPException(status_code=400, detail="cloth_type must be upper, lower, overall or shoes")
    person_img = resize_and_crop(Image.open(io.BytesIO(person.file.read())).convert("RGB"), (WIDTH, HEIGHT))
    cloth_raw = Image.open(io.BytesIO(cloth.file.read())).convert("RGB")
    with gpu_lock:
        cloth_clean, how = clean_garment(cloth_raw, cloth_type)
        cloth_img = resize_and_padding(cloth_clean, (WIDTH, HEIGHT))
        if cloth_type == "shoes":
            # Experimental: CatVTON is trained on clothes, so shoes may come out rough.
            feet, _ = find_parts(person_img, "shoes", min_share=0.0008)
            if feet is None:
                raise HTTPException(status_code=422, detail="Couldn't find your feet in the photo.")
            mask = Image.fromarray((feet * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(25))
        else:
            try:
                mask = automasker(person_img, cloth_type)["mask"]
            except Exception as e:
                raise HTTPException(status_code=422, detail=f"Couldn't find a person in the photo: {e}")
        mask = mask_processor.blur(mask, blur_factor=9)
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
        result = pipeline(image=person_img, condition_image=cloth_img, mask=mask,
                          num_inference_steps=max(10, min(int(steps), 50)), guidance_scale=2.5,
                          generator=generator)[0]
        blocked = is_blocked(result)
        print(f"try-on: {cloth_type}, garment photo {how}, {steps} steps")
    if blocked:
        # 451 = "unavailable for legal reasons": Adaa reads this as "the safety filter blocked it".
        raise HTTPException(status_code=451, detail="The safety filter blocked this result.")
    buf = io.BytesIO()
    result.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png")


nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)

# A free Cloudflare "quick tunnel" gives this Colab session a public https link.
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("/content/cloudflared", 0o755)
log = open("/content/tunnel.log", "w")
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found[0]
        break

print("=" * 60)
print("Paste these into Adaa > Preferences > Try-on server:")
print("  Server link :", url or "(not ready yet; run this cell again)")
print("  Token       :", TOKEN)
print("=" * 60)
print("Keep this tab open while you use Adaa.")

## Troubleshooting
- **"No GPU"** → *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
- **Out of memory** → *Runtime → Restart session*, then *Run all*.
- **Adaa says the server is unreachable** → Colab may have disconnected. Run cell 5 again (or *Run all*) and paste the **new** link and token into Adaa.
- **An error in cell 2 or 3** → copy the error text and send it to your developer. Colab updates its preinstalled packages often, so a version may need adjusting.